In [1]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_excel("../muestra_terra/Terra_example.xlsx")

In [24]:
import re
from collections import Counter

# Asegúrate de tener cargado tu DataFrame como `df`
requests = df['Request'].dropna().astype(str).tolist()

# Patrones para detectar secciones dentro del texto
section_patterns = [
    r"in the ([A-Z\s]+?) section",
    r"in the ([A-Z\s]+?) page",
    r"on the ([A-Z\s]+?) section",
    r"on the ([A-Z\s]+?) page",
    r"in ([A-Z\s]+?) section",
    r"on ([A-Z\s]+?) page"
]

# Extraer y normalizar
found_sections = []
for text in requests:
    for pattern in section_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        found_sections.extend([m.strip().title() for m in matches])

# Contar ocurrencias
section_counts = Counter(found_sections)
print(section_counts.most_common(10))

[('Who We Are', 5), ('The Who We Are', 4), ('The', 1), ('Keeping The Header Copy The Same On Both', 1)]


In [81]:
import pandas as pd
import random
from faker import Faker
from transformers import pipeline, set_seed
from datetime import date, datetime, timedelta

fake = Faker()
set_seed(42)

request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]

sections_base = ['header', 'footer', 'product page', 'sign-up form', 'testimonial section', 'pricing table', 'FAQ block']
modifiers = ['main', 'mobile version', 'desktop view', 'CTA zone', 'hover state', 'responsive layout']
sections = [f"{mod} of {base}" for base in sections_base for mod in random.sample(modifiers, 2)]

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

introductions = [
    "Hey team, this is {requester} from {company}.",
    "Hi there! {requester} from {company} here.",
    "{requester} here – I'm with {company}, working on {project}.",
    "Hi! I’m looking at {project} from {company}.",
    "Quick note from {requester} ({company}).",
    "{requester}, {company} – small issue I noticed.",
    "Hi all, reviewing {project}, found something.",
    "Heads up! {requester} from {company} noticed this.",
    "Reporting a detail in {project} – {requester}, {company}.",
]

tech_terms_by_type = {
    'Copy Revision': ['tone of voice', 'CTA', 'microcopy', 'UX writing'],
    'Design Issues': ['UI consistency', 'alignment', 'spacing', 'responsive layout', 'mobile-first'],
    'Requested Change': ['client feedback', 'scope update', 'functional change', 'new spec'],
    'New Item': ['feature', 'widget', 'module', 'custom component', 'new interaction']
}
buzzwords_common = ['UI', 'UX', 'layout', 'interaction', 'feedback', 'component', 'structure', 'copy', 'text', 'spacing']

generic_templates = [
    "Noticed something odd in {section}.",
    "Please check what's happening in {section}.",
    "Potential issue detected in {section}.",
    "Let’s improve {section}.",
    "Can we review the {section} again?"
]

templates = {
    'Copy Revision': [
        "Please revise the copy in {section}.",
        "Copy update needed for {section}.",
        "Text correction required in {section}.",
        "Rewriting needed for {section}.",
        "The content in {section} needs improvement."
    ],
    'Design Issues': [
        "Design problem found in {section}.",
        "Layout issue detected in {section}.",
        "Inconsistency in design of {section}.",
        "Visual bug in {section} needs fixing.",
        "Improve visual design in {section}."
    ],
    'Requested Change': [
        "Client requested a change in {section}.",
        "Requested modification for {section}.",
        "Adjustment needed in {section}.",
        "Update required following feedback in {section}.",
        "Changes needed according to client input in {section}."
    ],
    'New Item': [
        "New component needed in {section}.",
        "Add a new feature to {section}.",
        "Create something new in {section}.",
        "New item requested for {section}.",
        "Build a new section in {section}."
    ]
}

urgency_phrases = {
    'Low': ["", "No rush at all.", "Whenever there's time.", "Non-blocking."],
    'Normal': ["No big deal, but worth checking.", "Should go in this sprint.", "Minor but relevant."],
    'High': ["Pretty important, please prioritize.", "Client’s expecting this.", "Would prefer ASAP."],
    'Urgent': ["This broke things.", "Team is blocked.", "Immediate fix needed.", "On the critical path!"]
}

def generate_request(request_type, section, priority, company, requester, project, page):
    section_variants = [f"the {section}", f"section '{section}'", f"'{section}' section", f"area: {section}", f"in {section}"]
    section_phrase = random.choice(section_variants)

    if random.random() < 0.3:
        base_template = random.choice(generic_templates)
    else:
        base_template = random.choice(templates.get(request_type, ["Update {section}."]))

    base = base_template.replace("{section}", section_phrase)

    buzzword_pool = tech_terms_by_type.get(request_type, []) + buzzwords_common
    buzzword = random.choice(buzzword_pool) if buzzword_pool and random.random() < 0.5 else ""
    if buzzword:
        base += f" Consider {buzzword}."

    urgency = random.choice(urgency_phrases.get(priority, [""]))
    punctuation = random.choice([".", "!", "..."])

    intro = ""
    if random.random() < 0.7:
        intro_template = random.choice(introductions)
        intro = intro_template.format(
            requester=requester,
            company=company,
            project=project,
            page=page
        ) + " "

    return intro + base + urgency + punctuation

num_clients = 23
client_names = list({fake.company() for _ in range(num_clients * 2)})[:num_clients]
random.shuffle(client_names)
clients = {}
for i, name in enumerate(client_names):
    if i < 5:
        clients[name] = 'large'
    elif i < 15:
        clients[name] = 'medium'
    else:
        clients[name] = 'small'
size_weights = {'large': 5, 'medium': 3, 'small': 1}
weighted_clients = []
for client, size in clients.items():
    weighted_clients.extend([client] * size_weights[size])

project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
projects_per_client = {}
for client, size in clients.items():
    num_projects = {'large': random.randint(5, 8), 'medium': random.randint(3, 5), 'small': random.randint(1, 2)}[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        project_name = f"{base} {suffix} – {color}"
        projects.add(project_name)
    projects_per_client[client] = list(projects)

start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)
project_date_bases = {}
for project_list in projects_per_client.values():
    for project in project_list:
        project_date_bases[project] = fake.date_between(start_date=start_date, end_date=end_date)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

weighted_browsers = ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 + ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

data = []
for _ in range(10000):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)
    priority = random.choice(priority_weights_by_type[request_type])
    page = generate_client_page(project)
    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    real_time = max(1, round(random.normalvariate(estimated_time, 0.3)))
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]

    if status_value == "Complete":
        real_time = max(1, round(random.normalvariate(estimated_time, estimated_time * 0.3)))
    elif status_value == "In Progress":
        real_time = round(estimated_time * random.uniform(1.0, 1.5))
    else:
        real_time = 0

    requester = fake.name()

    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": requester,
        "Request Type": request_type,
        "Priority": priority,
        "Request": generate_request(request_type, section, priority, client, requester, project, page),
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })

df = pd.DataFrame(data)


In [83]:
import pandas as pd
import random
from faker import Faker
from datetime import date, datetime, timedelta

# Inicialización
fake = Faker()

# Configuración de empresas con atributos
client_attributes = pd.read_csv("tiering_sample\client_strategic_fit.csv", index_col=0)  # Carga desde un CSV si lo usas externo

client_attributes['Company'] = [fake.company() for _ in range(len(client_attributes))]

# Configurar categorías
request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
sections = ['hero section', 'footer', 'about us page', 'contact form', 'pricing table', 'FAQ block']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

# Introducciones e ideas de lenguaje
introductions = [
    "Hey team, this is {requester} from {company}.",
    "Hi there! {requester} from {company} here.",
    "{requester} here – I'm with {company}, working on {project}.",
    "Hi! I’m looking at {project} from {company}.",
    "Quick note from {requester} ({company}).",
    "{requester}, {company} – small issue I noticed.",
    "Hi all, reviewing {project}, found something.",
    "Heads up! {requester} from {company} noticed this.",
    "Reporting a detail in {project} – {requester}, {company}.",
]


tech_terms_by_type = {
    'Copy Revision': ['tone of voice', 'CTA', 'microcopy', 'UX writing'],
    'Design Issues': ['UI consistency', 'alignment', 'spacing', 'responsive layout', 'mobile-first'],
    'Requested Change': ['client feedback', 'scope update', 'functional change', 'new spec'],
    'New Item': ['feature', 'widget', 'module', 'custom component', 'new interaction']
}
buzzwords_common = ['UI', 'UX', 'layout', 'interaction', 'feedback', 'component', 'structure', 'copy', 'text', 'spacing']

def generate_request(request_type, section, priority, company, requester, project, page):
    section_variants = [f"the {section}", f"section '{section}'", f"'{section}' section", f"area: {section}", f"in {section}"]
    section_phrase = random.choice(section_variants)

    urgency_phrases = {
        'Low': ["", " Not urgent.", " Can be handled later."],
        'Normal': [" Please address this soon.", " Handle this in the upcoming days.", " This should be scheduled accordingly."],
        'High': [" This is a high priority item.", " Should be prioritized.", " Requires prompt attention."],
        'Urgent': [" URGENT! This blocks progress.", " Immediate action required.", " Needs to be resolved ASAP."]
    }

    templates = {
    'Copy Revision': [
        "Please revise the copy in {section}.",
        "Copy update needed for {section}.",
        "Text correction required in {section}.",
        "Rewriting needed for {section}.",
        "The content in {section} needs improvement."
    ],
    'Design Issues': [
        "Design problem found in {section}.",
        "Layout issue detected in {section}.",
        "Inconsistency in design of {section}.",
        "Visual bug in {section} needs fixing.",
        "Improve visual design in {section}."
    ],
    'Requested Change': [
        "Client requested a change in {section}.",
        "Requested modification for {section}.",
        "Adjustment needed in {section}.",
        "Update required following feedback in {section}.",
        "Changes needed according to client input in {section}."
    ],
    'New Item': [
        "New component needed in {section}.",
        "Add a new feature to {section}.",
        "Create something new in {section}.",
        "New item requested for {section}.",
        "Build a new section in {section}."
    ]
}

    tech_terms = tech_terms_by_type.get(request_type, [])
    buzzword = random.choice(tech_terms) if tech_terms and random.random() < 0.5 else ""
    section_phrase = random.choice(section_variants)    

    if random.random() < 0.3:
        base_template = random.choice(generic_templates)
    else:
        base_template = random.choice(templates.get(request_type, ["Update {section}."]))

    base = base_template.replace("{section}", section_phrase)

    tech_terms = tech_terms_by_type.get(request_type, [])
    buzzword_pool = tech_terms + buzzwords_common
    buzzword = random.choice(buzzword_pool) if buzzword_pool and random.random() < 0.5 else ""
    if buzzword:
        base += f" Consider {buzzword}."
    urgency = random.choice(urgency_phrases.get(priority, [""]))

    punctuation = random.choice([".", "!", "..."])

    intro = ""
    if random.random() < 0.7:
        intro_template = random.choice(introductions)
        intro = intro_template.format(
            requester=requester,
            company=company,
            project=project,
            page=page
        ) + " "

    return intro + base + urgency + punctuation

# Estimación de tiempo por tipo
estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

# Generador de página web
def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

# Browser y device
weighted_browsers = ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 + ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

# Fechas base por proyecto
start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

# Lógica por cliente
project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
project_date_bases = {}
projects_per_client = {}
weighted_clients = []

for _, row in client_attributes.iterrows():
    company = row['Company']
    strategic_fit = row['ClientStrategicFit']
    size = 'large' if strategic_fit >= 4.5 else 'medium' if strategic_fit >= 3.0 else 'small'
    num_projects = {
        'large': random.randint(5, 8),
        'medium': random.randint(3, 5),
        'small': random.randint(1, 2)
    }[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        projects.add(f"{base} {suffix} – {color}")
    projects = list(projects)
    projects_per_client[company] = projects
    for p in projects:
        project_date_bases[p] = fake.date_between(start_date=start_date, end_date=end_date)
    weighted_clients.extend([company] * int(strategic_fit * 2))

# Generación de datos
n = 10000
data = []
for _ in range(n):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)
    priority = random.choice(priority_weights_by_type[request_type])
    page = generate_client_page(project)

    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]

    if status_value == "Complete":
        real_time = max(1, round(random.normalvariate(estimated_time, estimated_time * 0.3)))
    elif status_value == "In Progress":
        real_time = round(estimated_time * random.uniform(1.0, 1.5))
    else:
        real_time = 0

    requester = fake.name()
    request = generate_request(request_type, section, priority, client, requester, project, page)

    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": requester,
        "Request Type": request_type,
        "Priority": priority,
        "Request": request,
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })


# Convertir a DataFrame
df = pd.DataFrame(data)

# Añadir el strategic
mapping = client_attributes.set_index('Company')['ClientStrategicFit']
df['ClientStrategicFit'] = df['Company'].map(mapping)


In [6]:
import pandas as pd
import random
from faker import Faker
from datetime import date, datetime, timedelta

# Inicialización
fake = Faker()

# Configuración de empresas con atributos
client_attributes = pd.read_csv("tiering_sample\client_strategic_fit.csv", index_col=0)  # Carga desde un CSV si lo usas externo

client_attributes['Company'] = [fake.company() for _ in range(len(client_attributes))]

mapping = client_attributes.set_index('Company')['ClientStrategicFit']

# Configurar categorías
request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Normal', 'High', 'Urgent']
sections = ['hero section', 'footer', 'about us page', 'contact form', 'pricing table', 'FAQ block']
status_choices = ["Complete", "In Progress", "In Queue"]
status_weights = [0.2, 0.5, 0.3]

priority_weights_by_type = {
    'Copy Revision':      ['Low'] * 6 + ['Normal'] * 3 + ['High'] * 1,
    'Design Issues':      ['Normal'] * 4 + ['High'] * 3 + ['Urgent'] * 2 + ['Low'] * 1,
    'Requested Change':   ['Normal'] * 3 + ['High'] * 4 + ['Urgent'] * 2 + ['Low'] * 1,
    'New Item':           ['High'] * 4 + ['Urgent'] * 4 + ['Normal'] * 2,
}

# Introducciones e ideas de lenguaje
introductions = [
    "Hey team, this is {requester} from {company}.",
    "Hi there! {requester} from {company} here.",
    "{requester} here – I'm with {company}, working on {project}.",
    "Hi! I’m looking at {project} from {company}.",
    "Quick note from {requester} ({company}).",
    "{requester}, {company} – small issue I noticed.",
    "Hi all, reviewing {project}, found something.",
    "Heads up! {requester} from {company} noticed this.",
    "Reporting a detail in {project} – {requester}, {company}.",
]


tech_terms_by_type = {
    'Copy Revision': ['tone of voice', 'CTA', 'microcopy', 'UX writing'],
    'Design Issues': ['UI consistency', 'alignment', 'spacing', 'responsive layout', 'mobile-first'],
    'Requested Change': ['client feedback', 'scope update', 'functional change', 'new spec'],
    'New Item': ['feature', 'widget', 'module', 'custom component', 'new interaction']
}
buzzwords_common = ['UI', 'UX', 'layout', 'interaction', 'feedback', 'component', 'structure', 'copy', 'text', 'spacing']

def generate_request(request_type, section, priority, company, requester, project, page):
    section_variants = [f"the {section}", f"section '{section}'", f"'{section}' section", f"area: {section}", f"in {section}"]
    section_phrase = random.choice(section_variants)

    generic_templates = [
    "Noticed something odd in {section}.",
    "Please check what's happening in {section}.",
    "Potential issue detected in {section}.",
    "Let’s improve {section}.",
    "Can we review the {section} again?"
    ]

    urgency_phrases = {
        'Low': ["", " Not urgent.", " Can be handled later."],
        'Normal': [" Please address this soon.", " Handle this in the upcoming days.", " This should be scheduled accordingly."],
        'High': [" This is a high priority item.", " Should be prioritized.", " Requires prompt attention."],
        'Urgent': [" URGENT! This blocks progress.", " Immediate action required.", " Needs to be resolved ASAP."]
    }

    templates = {
    'Copy Revision': [
        "Please revise the copy in {section}.",
        "Copy update needed for {section}.",
        "Text correction required in {section}.",
        "Rewriting needed for {section}.",
        "The content in {section} needs improvement."
    ],
    'Design Issues': [
        "Design problem found in {section}.",
        "Layout issue detected in {section}.",
        "Inconsistency in design of {section}.",
        "Visual bug in {section} needs fixing.",
        "Improve visual design in {section}."
    ],
    'Requested Change': [
        "Client requested a change in {section}.",
        "Requested modification for {section}.",
        "Adjustment needed in {section}.",
        "Update required following feedback in {section}.",
        "Changes needed according to client input in {section}."
    ],
    'New Item': [
        "New component needed in {section}.",
        "Add a new feature to {section}.",
        "Create something new in {section}.",
        "New item requested for {section}.",
        "Build a new section in {section}."
    ]
}

    tech_terms = tech_terms_by_type.get(request_type, [])
    buzzword = random.choice(tech_terms) if tech_terms and random.random() < 0.5 else ""
    section_phrase = random.choice(section_variants)    

    if random.random() < 0.3:
        base_template = random.choice(generic_templates)
    else:
        base_template = random.choice(templates.get(request_type, ["Update {section}."]))

    base = base_template.replace("{section}", section_phrase)

    tech_terms = tech_terms_by_type.get(request_type, [])
    buzzword_pool = tech_terms + buzzwords_common
    buzzword = random.choice(buzzword_pool) if buzzword_pool and random.random() < 0.5 else ""
    if buzzword:
        base += f" Consider {buzzword}."
    urgency = random.choice(urgency_phrases.get(priority, [""]))

    punctuation = random.choice([".", "!", "..."])

    intro = ""
    if random.random() < 0.7:
        intro_template = random.choice(introductions)
        intro = intro_template.format(
            requester=requester,
            company=company,
            project=project,
            page=page
        ) + " "

    return intro + base + urgency + punctuation

# Estimación de tiempo por tipo
estimated_time_options = [1, 2, 3, 5, 8, 13]
estimated_time_weights = {
    'Copy Revision':      [0.5, 0.3, 0.1, 0.05, 0.03, 0.02],
    'Design Issues':      [0.1, 0.2, 0.3, 0.25, 0.1, 0.05],
    'Requested Change':   [0.05, 0.1, 0.3, 0.3, 0.15, 0.1],
    'New Item':           [0.02, 0.05, 0.1, 0.3, 0.3, 0.23],
}

# Generador de página web
def generate_client_page(company_name):
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"

# Browser y device
weighted_browsers = ["Chrome"] * 50 + ["Safari"] * 20 + ["Firefox"] * 15 + ["Edge"] * 8 + ["Brave"] * 3 + ["Opera"] * 2 + ["Internet Explorer"] * 1 + ["Samsung Internet"] * 1
devices_weighted = ["Desktop"] * 50 + ["Mobile"] * 45 + ["Tablet"] * 5

# Fechas base por proyecto
start_date = date(2020, 1, 1)
end_date = date(2025, 1, 1)

def generate_nearby_date(base_date, max_delta_days=45):
    delta = random.randint(-max_delta_days, max_delta_days)
    return (base_date + timedelta(days=delta)).strftime("%d/%m/%Y")

# Lógica por cliente
project_suffixes = ['Platform', 'Revamp', 'Redesign', 'Campaign', 'Website', 'Landing', 'Portal', 'Dashboard']
project_date_bases = {}
projects_per_client = {}
weighted_clients = []

for _, row in client_attributes.iterrows():
    company = row['Company']
    strategic_fit = row['ClientStrategicFit']
    size = 'large' if strategic_fit >= 4.5 else 'medium' if strategic_fit >= 3.0 else 'small'
    num_projects = {
        'large': random.randint(5, 8),
        'medium': random.randint(3, 5),
        'small': random.randint(1, 2)
    }[size]
    projects = set()
    while len(projects) < num_projects:
        base = fake.word().capitalize()
        suffix = random.choice(project_suffixes)
        color = fake.color_name().replace(" ", "")
        projects.add(f"{base} {suffix} – {color}")
    projects = list(projects)
    projects_per_client[company] = projects
    for p in projects:
        project_date_bases[p] = fake.date_between(start_date=start_date, end_date=end_date)
    weighted_clients.extend([company] * int(strategic_fit * 2))

# Tema prioridad
def biased_priority(request_type, strategic_fit):
    """Asigna prioridad con sesgo basado en el strategic_fit."""
    base_weights = priority_weights_by_type[request_type]
    
    # Define sesgo: más estratégico → + alta prioridad
    bias_factor = (strategic_fit - 3.0) / 2.0  # Rango aprox -1 a +1
    bias_factor = max(-1.0, min(1.0, bias_factor))  # Clamp por si acaso

    # Map priority index: Low=0, Normal=1, High=2, Urgent=3
    raw_weights = [0, 0, 0, 0]
    for p in priority_weights_by_type[request_type]:
        idx = priorities.index(p)
        raw_weights[idx] += 1
    total = sum(raw_weights)
    raw_weights = [w / total for w in raw_weights]  # Normaliza

    # Aplica sesgo: mueve peso de Low hacia High/Urgent y viceversa
    adjusted_weights = []
    for i, w in enumerate(raw_weights):
        if i == 0:  # Low
            w *= (1 - 0.4 * bias_factor)
        elif i == 3:  # Urgent
            w *= (1 + 0.4 * bias_factor)
        elif i == 2:  # High
            w *= (1 + 0.3 * bias_factor)
        elif i == 1:  # Normal
            w *= (1 - 0.1 * bias_factor)
        adjusted_weights.append(w)

    # Normaliza de nuevo
    total = sum(adjusted_weights)
    adjusted_weights = [w / total for w in adjusted_weights]

    return random.choices(priorities, weights=adjusted_weights, k=1)[0]

# More biass, aun más clasista
def generate_real_time(estimated_time, status, priority):
    if status == "Complete":
        if priority == "Urgent":
            real = random.normalvariate(estimated_time * 0.9, estimated_time * 0.2)
        elif priority == "High":
            real = random.normalvariate(estimated_time * 1.0, estimated_time * 0.25)
        elif priority == "Normal":
            real = random.normalvariate(estimated_time * 1.1, estimated_time * 0.35)
        else:  # Low
            real = random.normalvariate(estimated_time * 1.3, estimated_time * 0.5)
        return max(1, round(real))
    
    elif status == "In Progress":
        if priority == "Urgent":
            return round(estimated_time * random.uniform(0.9, 1.1))
        elif priority == "High":
            return round(estimated_time * random.uniform(1.0, 1.3))
        elif priority == "Normal":
            return round(estimated_time * random.uniform(1.0, 1.5))
        else:  # Low
            return round(estimated_time * random.uniform(1.1, 2.0))
    
    else:  # In Queue
        return 0


# Generación de datos
n = 10000
data = []
for _ in range(n):
    request_type = random.choice(request_types)
    section = random.choice(sections)
    client = random.choice(weighted_clients)
    project = random.choice(projects_per_client[client])
    base_date = project_date_bases[project]
    input_date = generate_nearby_date(base_date)

    # priority = random.choice(priority_weights_by_type[request_type])
    strategic_fit = mapping[client]
    priority = biased_priority(request_type, strategic_fit)

    page = generate_client_page(project)

    weights = estimated_time_weights[request_type]
    estimated_time = random.choices(estimated_time_options, weights=weights, k=1)[0]
    input_dt = datetime.strptime(input_date, "%d/%m/%Y")

    if input_dt.year < 2025:
        status_value = "Complete"
    else:
        status_value = random.choices(status_choices, weights=status_weights, k=1)[0]

    real_time = generate_real_time(estimated_time, status_value, priority)

    requester = fake.name()
    request = generate_request(request_type, section, priority, client, requester, project, page)

    data.append({
        "Company": client,
        "Project Name": project,
        "Input Date": input_date,
        "Status": status_value,
        "Requester": requester,
        "Request Type": request_type,
        "Priority": priority,
        "Request": request,
        "Device": random.choice(devices_weighted),
        "Browser": random.choice(weighted_browsers),
        "Page": page,
        "Estimated Time (tokens)": estimated_time,
        "Real Time": real_time,
    })


# Convertir a DataFrame
df = pd.DataFrame(data)

# Añadir el strategic
mapping = client_attributes.set_index('Company')['ClientStrategicFit']
df['ClientStrategicFit'] = df['Company'].map(mapping)


In [13]:
df = pd.read_csv("synthetic_example.csv")

In [7]:
df

,Company,Project Name,Input Date,Status,Requester,Request Type,Priority,Request,Device,Browser,Page,Estimated Time (tokens),Real Time,ClientStrategicFit
0,"Brown, Bell and Roach",Serve Redesign – CadetBlue,23/03/2022,Complete,Carrie Flores,Copy Revision,Normal,"Heads up! Carrie Flores from Brown, Bell and R...",Desktop,Opera,https://www.serveredesign–cadetblue.com,1,1,3.55
1,Matthews PLC,Throw Portal – Navy,06/10/2024,Complete,Lucas Hammond,New Item,Normal,Hi! I’m looking at Throw Portal – Navy from Ma...,Desktop,Firefox,https://www.throwportal–navy.com,5,5,2.60
2,"Foster, Howard and Beck",Federal Campaign – BurlyWood,05/03/2020,Complete,Robert Carroll,Copy Revision,Normal,Can we review the area: about us page again? C...,Mobile,Edge,https://www.federalcampaign–burlywood.com,1,2,4.20
3,Grimes and Sons,Need Dashboard – Coral,19/10/2020,Complete,Stephanie Hernandez,New Item,Urgent,Noticed something odd in section 'contact form...,Mobile,Safari,https://www.needdashboard–coral.com,8,6,4.70
4,"Gill, King and Henson",Discover Platform – HotPink,26/06/2021,Complete,Brian Rodriguez,Design Issues,High,Visual bug in 'hero section' section needs fix...,Mobile,Edge,https://www.discoverplatform–hotpink.com,2,2,4.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Grimes and Sons,Need Dashboard – Coral,21/11/2020,Complete,Julia Conrad,Copy Revision,Normal,"Julia Conrad here – I'm with Grimes and Sons, ...",Mobile,Chrome,https://www.needdashboard–coral.com,3,3,4.70
9996,"Gill, King and Henson",Past Platform – Red,03/12/2021,Complete,Darius Munoz,Design Issues,High,"Darius Munoz here – I'm with Gill, King and He...",Mobile,Edge,https://www.pastplatform–red.com,1,1,4.70
9997,Morgan PLC,Here Redesign – DarkSlateGray,13/04/2023,Complete,Vanessa Taylor,New Item,Urgent,"Hi all, reviewing Here Redesign – DarkSlateGra...",Desktop,Firefox,https://www.hereredesign–darkslategray.com,5,5,3.90
9998,Smith-Jackson,Physical Website – DarkGoldenRod,20/07/2020,Complete,Daniel Johnson,Requested Change,High,Reporting a detail in Physical Website – DarkG...,Mobile,Safari,https://www.physicalwebsite–darkgoldenrod.com,5,4,4.55


In [85]:
# Número de proyectos por empresa
proyectos_por_empresa = df[['Company', 'Project Name']].drop_duplicates()
proyectos_count = proyectos_por_empresa.groupby('Company').size().reset_index(name='Num Projects')

# Número de requests por proyecto
requests_count = df.groupby(['Company', 'Project Name']).size().reset_index(name='Num Requests')

# Merge para unir ambos
combined = requests_count.merge(proyectos_count, on='Company')

# Mostrar ordenado por empresa y luego por cantidad de requests
combined.sort_values(['Num Projects', 'Num Requests'], ascending=[False, False])

,Company,Project Name,Num Requests,Num Projects
5,Boone-Gonzalez,One Landing – DarkSlateGray,97,8
6,Boone-Gonzalez,Side Redesign – Purple,89,8
61,King PLC,Weight Landing – ForestGreen,84,8
8,Boone-Gonzalez,Trouble Revamp – NavajoWhite,82,8
2,Boone-Gonzalez,Always Dashboard – SeaGreen,74,8
...,...,...,...,...
70,"Powell, Christensen and Hernandez",Enter Portal – SeaShell,132,2
69,"Powell, Christensen and Hernandez",Describe Platform – LightSteelBlue,121,2
72,Rollins-Evans,Concern Revamp – DarkOliveGreen,259,1
71,Rodgers-Stephenson,Indicate Platform – DarkOliveGreen,243,1


In [86]:
#df2 = df[["Request Type" , "Status", "Input Date", "Requester", "Device", "Browser", "Request", "Page"]]

df.to_csv("synthetic_example.csv", index=False)

df.to_json("synthetic_example.json", orient="records", lines=True)